In [1]:
import json
import os
import pandas as pd
from sklearn.model_selection import train_test_split

INPUT_PATH = '/mnt/data/zwl/verl/data/path_only_80.jsonl'
OUTPUT_DIR = '/mnt/data/zwl/verl/data/rl/'

data = []
with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

print(f"Loaded {len(data)} records.")

instruction_following = "\n\nPlease reason step by step, and put your final answer within \\boxed{}."

def format_verl_record(row):
    problem = row.get("problem", row.get("input", row.get("question", "")))
    ground_truth = row.get("ground_truth", row.get("answer", row.get("expected", row.get("output", ""))))
    
    q_text = str(problem).strip()
    if instruction_following not in q_text:
         q_text += instruction_following
         
    return {
        "data_source": "mixed_dat",
        "prompt": [{"role": "user", "content": q_text}],
        "reward_model": {"ground_truth": str(ground_truth).strip()},
        "extra_info": {"prompt": q_text}
    }

formatted_data = [format_verl_record(row) for row in data]
df_formatted = pd.DataFrame(formatted_data)

train_df, val_df = train_test_split(df_formatted, test_size=0.1, random_state=42)
print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

train_path = os.path.join(OUTPUT_DIR, 'qwen3_4b_grpo_mixed_train.parquet')
val_path = os.path.join(OUTPUT_DIR, 'qwen3_4b_grpo_mixed_val.parquet')

train_df.to_parquet(train_path, index=False)
val_df.to_parquet(val_path, index=False)

print(f"✅ Successfully saved dataset.")
print(f"   Train: {train_path}")
print(f"   Val:   {val_path}")

Loaded 80 records.
Train size: 72
Val size: 8
✅ Successfully saved dataset.
   Train: /mnt/data/zwl/verl/data/rl/qwen3_4b_grpo_mixed_train.parquet
   Val:   /mnt/data/zwl/verl/data/rl/qwen3_4b_grpo_mixed_val.parquet
